## Sampling with Fixed Lambda

### Data

Test lambda = 0.01, 0.1, 1, 10, 100?

In [68]:
lb_val = 0.01
desc = "lbda"+str(lb_val)

# keep this constant
sampler_name = "emcee"
numwalkers = 50
numsteps = 200
numburn = 20
a_val = 2

In [69]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import bilby
from bilby.core.utils import random
import json
import scipy.special
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# where / how to save files
label = f"{sampler_name}_{desc}"
outdir = f"{numwalkers}walkers"
bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)
random.seed(123)

In [70]:
### --- to load diabetes data ---
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X = diabetes.data

selected_features = ['bmi', 'bp', 's1']
X = X[selected_features]

label_names = diabetes.feature_names
y = diabetes.target

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain) # fit and scale training data
Xtest = scaler.transform(Xtest) # scale test data

### MCMC

In [71]:
# helper function to compute the RBF kernel
def rbf_kernel(X, ell, sigma_gp):
    N = X.shape[0]
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            diff = (X[i] - X[j]) / ell
            K[i, j] = sigma_gp**2 * np.exp(-0.5 * np.dot(diff, diff))
    return K

In [72]:
# custom likelihood for penalized GP regression

class PenalizedGPLikelihood(bilby.Likelihood):
    def __init__(self, X, y):
        # store data
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # define parameters
        parameters = {}

        for i in range(self.X.shape[1]):
            parameters[f"ell{i}"] = None # lengthscales (ells)     
            parameters[f"nu{i}"] = None # nu: random variable for beta calculation
            parameters[f"zeta{i}"] = None # zeta: random variable for tau calculation

        # noise parameters
        parameters["inv_sigma_noise"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        parameters["inv_sigma_gp"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        # parameters["lambda2"] = None # lambda_squared 

        super().__init__(parameters=parameters)


    def log_likelihood(self):

        # extract parameters
        inv_sigma_noise = self.parameters["inv_sigma_noise"]
        inv_sigma_gp = self.parameters["inv_sigma_gp"]
        ells = np.array([self.parameters[f"ell{i}"] for i in range(self.X.shape[1])])
        # lbda2 = self.parameters["lambda2"] # sample lambda squared

        # --- fixed lambda value ---
        lbda2 = lb_val**2

        # --- transformers ---
        nus = np.array([self.parameters[f"nu{i}"] for i in range(self.X.shape[1])]) # random var for beta
        zetas = np.array([self.parameters[f"zeta{i}"] for i in range(self.X.shape[1])]) 
        
        # calculate n and p
        n = self.X.shape[0]
        p = self.X.shape[1]

        # --- transform sampled gamma to desired inverse gamma ---
        sigma_gp = 1/inv_sigma_gp if inv_sigma_gp is not None else None
        sigma_noise = 1/inv_sigma_noise if inv_sigma_noise is not None else None

        # --- transform zetas to tau^2 ---
        # each tau^2 \sim expoential(\lambda^2 / 2)
        if (zetas is not None and lbda2 is not None):
            tau_sqs = (2 * zetas) / lbda2

        # --- transform nu to beta ---                   
        if (nus is not None and tau_sqs is not None and sigma_noise is not None):
            betas = nus * np.sqrt(tau_sqs) * sigma_noise 
        
        C = rbf_kernel(self.X, ells, sigma_gp) # calculate covariance matrix C
        residuals = self.y - self.X @ betas  # calculate residuals
        D = np.diag(tau_sqs) # calculate diagonal matrix D

        # compute ahead of time
        sigma2_D = sigma_noise**2 * D
        

        # --- log likelihood components ---
        log_lik_gp = (
            -0.5 * n * np.log(2 * np.pi) 
            -0.5 * np.linalg.slogdet(C + sigma_noise**2 * np.eye(n))[1]
            -0.5 *(residuals).T @ np.linalg.solve(C + sigma_noise**2 * np.eye(n), residuals))

        # rewritten to avoid inversion of D
        if np.any(sigma2_D == 0):
            log_lik_beta = (
                -0.5 * p * np.log(2 * np.pi)
                -0.5 * np.linalg.slogdet(sigma_noise**2 * D)[1]
                -0.5 * betas.T @ np.linalg.solve(sigma_noise**2 * D, betas)
            )
        else:  
            inv_sigma2_D = 1 / sigma2_D
            quadratic = np.dot((betas.ravel()**2), inv_sigma2_D)
            logdet = np.sum(np.log(sigma2_D))
            
            log_lik_beta = (
                -0.5 * p * np.log(2 * np.pi)
                -0.5 * logdet
                -0.5 * quadratic)
        
        return log_lik_gp + log_lik_beta


In [73]:
# make priors
priors = dict()

priors["inv_sigma_noise"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_noise")  
priors["inv_sigma_gp"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_gp")  
# priors["lambda2"] = bilby.core.prior.Gamma(1.0, 1.78, name="lambda2") # positive

for i in range(Xtrain.shape[1]):
    priors[f"zeta{i}"] = bilby.core.prior.Exponential(1, f"zeta{i}") # exponential with mean 1 = rate 1
    priors[f"ell{i}"] = bilby.core.prior.LogNormal(0, 1, f"ell{i}") # define log-normal priors for each lengthscale
    priors[f"nu{i}"] = bilby.core.prior.Normal(0,1, f"nu{i}") # standard normal for nu

# define the likelihood function that we defined earlier
likelihood = PenalizedGPLikelihood(
    X = Xtrain,
    y = ytrain)

In [ ]:
# run MCMC sampler
result = bilby.run_sampler(
    likelihood=likelihood, # likelihood function
    priors=priors, # prior distributions
    sampler=sampler_name, # other options for mcmc are emcee, zeus, , pyemcee, ptemcee, bilby-mcmc 
    nwalkers = numwalkers , # need > 2 x number of parameters
    nsteps = numsteps,
    nburn = numburn,
    sampler_kwargs = dict(a=a_val),
    outdir=outdir,
    label=label)

In [ ]:
result.posterior

In [76]:
# create new dataframe to store results
result_tab = result.posterior.copy()
lbda2 = lb_val**2

# calculate tau_sq from samples
for i in range(X.shape[1]):
    zetas = result_tab[f"zeta{i}"].values
    tau_sqs = (2 * zetas) / lbda2
    result_tab[f"tau_sq{i}"] = tau_sqs

result_tab["lambda"] = lb_val

# calculate beta from samples
for i in range(X.shape[1]):
    nus = result_tab[f"nu{i}"].values
    result_tab["sigma_noise"] = 1 / result_tab["inv_sigma_noise"].values
    tau_sqs = result_tab[f"tau_sq{i}"].values
    betas = nus * np.sqrt(tau_sqs) * result_tab["sigma_noise"].values
    result_tab[f"beta{i}"] = betas


clean_tab = result_tab.copy()
drop_cols = [f"zeta{i}" for i in range(X.shape[1])] + [ "inv_sigma_noise", "inv_sigma_gp"] + [f"nu{i}" for i in range(X.shape[1])]
clean_tab = clean_tab.drop(columns=drop_cols) # drop unneeded columns
clean_tab = clean_tab.reindex(sorted(clean_tab.columns), axis=1) # sort columns alphabetically
    

In [ ]:
type(result_tab)


In [78]:
result.plot_walkers()


In [ ]:
# calculate acceptance fraction

import pickle

with open(f"{numwalkers}walkers/{sampler_name}_{sampler_name}{id}_{desc}/sampler.pickle", "rb") as f:
    sampler = pickle.load(f)



af = sampler.acceptance_fraction
print("Per-walker acceptance fractions:", af)
print("Mean acceptance fraction:", af.mean())


In [ ]:
clean_tab.median()

In [ ]:
fig, axes = plt.subplots(X.shape[1], 1, figsize=(10, 3*X.shape[1]), sharex=True)

for i in range(X.shape[1]):
    ax = axes[i]
    traces = clean_tab[f"beta{i}"].to_numpy().reshape(50, 9000).T
    ax.plot(traces, alpha=0.5, linewidth=0.5)  # matplotlib will plot each walker as a line
    ax.set_ylim([-60, 60])
    ax.set_title(f"Trace plot for beta{i}")
    ax.set_ylabel(f"beta{i}")

axes[-1].set_xlabel("Step")
plt.tight_layout()
plt.show()